In [1]:
from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.ligsaf_utilities import create_rcf_utilities_system

In [2]:
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


In [3]:
# Code just to increase the number of display units for the various components
tmo.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
tmo.MultiStream.display_units.N = 40  
bst.Stream.display_units.N = 40       # Increasing number of display units to see all components of streams 
bst.MultiStream.display_units.N = 40  

In [4]:

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_Oil)
monomer_purification_sys = create_monomer_purification_system(ins=F.Purified_RCF_Oil)
rcf_oil_purification_sys.simulate()

BT, WWT, gas_mixer = create_rcf_utilities_system()

rcf_combined_system = bst.System(
    'Combined_RCF_System',
    path=(rcf_system, rcf_oil_purification_sys, WWT),
    facilities=[gas_mixer, BT],
)
rcf_combined_system.simulate()
rcf_combined_system.show()


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: PUMP101> no pump type available at current power (1.96e+03 hp), head (3.23e+03 ft), kinematic viscosity (6.01e-07 m2/s), and NPSH (3.96 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:409: CostWarning: <SolvolysisReactor: RCF103_S> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: PUMP108> power (0 hp) i

System: Combined_RCF_System
Highest convergence error among components in recycle
streams {HX117-0, PUMP112-0} after 1 loops:
- flow rate   1.86e-10 kmol/hr (0.012%)
- temperature 1.84e-03 K (0.00059%)
ins...
[0] RCF_Catalyst  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): NiC  2.28
[1] WW_11  from  LiquidsSplitCentrifuge-CENT303
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow: 0
[2] WW_12  from  MultiStageMixerSettlers-LLE300
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow: 0
[3] air_lagoon  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): N2  530
                    O2  131
[4] caustic  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  8.26
                    NaOH   3.72
[5] Poplar_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water     925
                    Sucrose   0.243
                    Extract   7.4
                    Acetate   48.6
                    Ash       1e+03
                    Lignin  

c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <LiquidsMixingTank> Vertical vessel weight (77.81 lb) is out of bounds (4200 to 1e+06 lb) for cost correlation
  return method(pressure, diameter, length)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <LiquidsMixingTank> Vertical vessel length (1.257 ft) is out of bounds (12 to 40 ft) for cost correlation
  return method(pressure, diameter, length)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <LiquidsSettler> Horizontal vessel diameter (2.793 ft) is out of bounds (3 to 21 ft) for cost correlation
  return method(pressure, diameter, length)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\design_tools\pressure_vessel.py:104: CostWarning: <Flash: FLASH201> Vertical vessel length (10.5 ft) is out of boun

In [5]:
chems

CompiledChemicals([Water, Ethanol, AceticAcid, Furfural, Glycerol, H2SO4, NH3, LacticAcid, SuccinicAcid, P4O10, Lime, HNO3, NH4OH, Denaturant, DAP, AmmoniumAcetate, AmmoniumSulfate, NaNO3, Oil, HMF, N2, O2, CH4, H2S, SO2, CO2, NO2, NO, CO, Glucose, Xylose, Sucrose, CaSO4, Mannose, Galactose, Arabinose, CellulaseNutrients, Extract, Acetate, Tar, Ash, NaOH, Lignin, SolubleLignin, GlucoseOligomer, GalactoseOligomer, MannoseOligomer, XyloseOligomer, ArabinoseOligomer, Z_mobilis, T_reesei, Biomass, Cellulose, Protein, Enzyme, Glucan, Xylan, Xylitol, Cellobiose, CSL, DenaturedEnzyme, Arabinan, Mannan, Galactan, WWTsludge, Cellulase, Methanol, Hydrogen, Methane, Hexane, EthylAcetate, propylcyclohexane, Ethylene, Butene, Hex-1-ene, Dec-1-ene, Octadec-1-ene, Butane, Decane, Octadecane, Syndol, Nickel_SiAl, CobaltMolybdenum, Coal, NiC, ActivatedCarbon, Propylguaiacol, Propylsyringol, Syringaresinol, G_Dimer, S_Oligomer, G_Oligomer])


In [6]:
# Different flows for the two monomers - strange, probably have to do with the scaling function used and the monomer yields. 
# This is why constantly checking the system is crucial

In [7]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
    # Hydrodeoxygenation reactions
    solvolysis_rxn = bst.Reaction(
        'Lignin -> SolubleLignin', reactant='Lignin',
        X=solvolysis_parameters['Delignification'],
        basis='wt', correct_atomic_balance=False
    )
    hydrodeoxygenation_rxn = bst.ParallelReaction([
        bst.Reaction('Propylguaiacol,l -> propyl_cyclohexane,l', reactant='Propylguaiacol', phases='lg',
                     X=1.0, basis='wt', correct_atomic_balance=False),
        bst.Reaction('Propylguaiacol,l -> propyl_cyclohexane,l', reactant='Propylguaiacol', phases='lg',
                     X=1.0, basis='wt', correct_atomic_balance=False),
    ])


In [ ]:
import biosteam as bst, numpy as np
from math import ceil

from typing import Optional
from biosteam.units.design_tools import (
    PressureVessel, 
)
 

class HydrodeoxygenationReactor(bst.Unit, bst.units.design_tools.PressureVessel):

    """
    Batch reactor for HDO of lignin oil
    Can take in complete RCF Oil, hence no need to isolate monomers 
    
    
    References
    ----------------------------------------------------------------------------------
        [1] Bruno Pandalone., et a;.
        "Optimum Lignin Oil - Finding the Most Suitable Feedstock to Replace Cycloalkanes in Sustainable Aviation Fuel (SAF)"
        ChemSusChem. 2025. 18(11). https://doi.org/10.1002/cssc.202402531
   -----------------------------------------------------------------------------------

    
    """



    _F_BM_default = {'Horizontal pressure vessel': 3.05,
                     'Vertical pressure vessel': 4.16,
                     'Platform and ladders': 1.}                   

    auxiliary_unit_names = (
        'pump_1', 'heat_exchanger_1'
    )

    _N_ins = 2
    _N_outs = 2
    
    _units = {**PressureVessel._units,
              'Pressure drop': 'bar',
              'Batch time': 'hr',
              'Turnaround time': 'hr',
              'Time on stream': 'hr',
              'Residence time': 'hr',
              'Total beds': "",
              'Beds in service': "",
              'Total volume': 'm3',
              'Reactor volume': 'm3',
              'Biomass volume per bed': 'm3',
              'Solvent volume per bed': 'm3',
              'Instantaneous loading': 'L/kg'}
    


    # Default operating temperature [K]
    T_default: float = 573.15                   # 300 C from [1]

    #: Default operating pressure [Pa]
    P_default:  float = 5e6                     # 5 MPa from [1]
    
    #: Default reaction time [hr]
    tau_default: float = 2                      # Total 2 hr reaction time, but could be higher (5 hr and 22 hr tested in [1])

    #: Default cleaning and unloading time (hr).
    tau_0_default: float  = 1                    # Assumed

    # Fixed bed configuration
    N_total_default: int =  3                      # Total beds (2 operating + 1 cleaning)

    N_working_default: int = 2                     # Beds operating at any time


    # Default catalyst bulk density [kg/m³]
    catalyst_bulk_density: float = 485            # Bulk density of the catalyst particle

    # Default free-space fraction of reactor volume
    free_frac_default: float = 0.10                # 10% kept free for gas disengagement / headspace
    
    # Default maximum vessel volume [m3]
    V_max_default: float = 600                     # Assumed

    # Aspect ratio (L/D of the reactor)
    aspect_ratio: float = 5.0                      # Assumed



    def _init(
            self,
            T: Optional[float] = None,
            P: Optional[float] = None,
            tau: Optional[float] = None,
            vessel_material: Optional[str] = None,
            vessel_type: Optional[str] = None,
            tau_0: Optional[float] = None,
            catalyst_bulk_density: Optional[float] = None,
            free_frac: Optional[float] = None,
            V_max: Optional[float] = None,
            V_max_limit: Optional[float] = None,
            aspect_ratio: Optional[float] = None,
            *,
            reaction_1            
            ):


        self.T = self.T_default if T is None else T
        self.P = self.P_default if P is None else P
        self.tau = self.tau_default if tau is None else tau
        self.vessel_material = 'Stainless steel 316' if vessel_material is None else vessel_material
        self.vessel_type = 'Vertical' if vessel_type is None else vessel_type
        self.tau_0 = self.tau_0_default if tau_0 is None else tau_0
        self.catalyst_bulk_density = self.catalyst_bulk_density if catalyst_bulk_density is None else catalyst_bulk_density
        self.free_frac      = self.free_frac_default      if free_frac      is None else free_frac
        self.V_max = self.V_max_default if V_max is None else V_max
        self.V_max_limit     = self.V_max_limit_default     if V_max_limit     is None else V_max_limit
        self.aspect_ratio          = self.aspect_ratio         if aspect_ratio          is None else aspect_ratio
        self.reaction_1 = reaction_1
        pump_1 = self.auxiliary('pump_1', bst.Pump, ins = self.ins[0])
        # heat_exchanger_1 = self.auxiliary('heat_exchanger_1', bst.HXutility, pump_1.outs[0])






    def _size_bed(self):


        cycle_time        = self.tau + self.tau_0                  


        # Total monomer flow
        total_monomer_flow = self.ins[0].F_vol                      # [m3/hr]

        V_theoretical = total_monomer_flow * self.tau               # [m3] Theoretical volume required
        
        V_actual = V_theoretical/(1+self.free_frac)                 # [m3] Actual volume required based on free fraction
        
        N_working = ceil(V_actual/self.V_max)                       # Number of working reactors based off maximum volume
        N_offline = ceil(N_working*(self.tau_0/cycle_time))         # Number of offline beds, calculated based off cleaning time and the total cycle time, rounded off to the next number
        N_total = N_working + N_offline                             # Total beds required
        V_total = N_total * self.V_max                              # Total volume required
        
        diameter = (4*self.V_max)/((self.aspect_ratio*np.pi)**1/3)
        length = self.aspect_ratio * inside_diameter


        return length, diameter, N_total, N_working, V_total

        
    def _run(self):
        monomers, hydrogen = self.ins
        effluent, = self.outs


        effluent.copy_like(monomers)
        self.reaction(effluent)


        used_solvent.P = self.P                                             # Assuming no P drop
        used_solvent.T = self.T                                             # Assuming isothermal operation
        


        solubilized_lignin = used_biomass.imass['SolubleLignin'] 
        used_solvent.imass['l', 'SolubleLignin'] += solubilized_lignin      # Soluble lignin dissolves in solvent effluent stream 
        used_biomass.imass['SolubleLignin'] = 0                             # No soluble lignin remaining in biomass (assuming 100% extraction efficiency)




        extractives = used_biomass.imass['Extract']                         # From Table S1 https://www.rsc.org/suppdata/d1/gc/d1gc01591e/d1gc01591e1.pdf,
                                                                            # it follows that the extractives component of poplar is 'extracted' in the solvent stream
        used_solvent.imass['l','Extract'] = (1-solvolysis_parameters['Extractives_retention'])*extractives
        used_biomass.imass['Extract'] = (solvolysis_parameters['Extractives_retention'])*extractives

        acetate = used_biomass.imass['Acetate']
        used_solvent.imass['l', 'Acetate'] =  acetate *(1-solvolysis_parameters['Acetate_retention']) # Assuming acetate dissolves as acetic acid with methanol,
                                                                             # BioSTEAM Chemicals assumes same properties for acetic acid and acetate, otherwise is acetate was a pseudocomponent, it might have still stayed in solid phase
        used_biomass.imass['Acetate'] = acetate*solvolysis_parameters['Acetate_retention']


        cellulose_mass = used_biomass.imass['Glucan']
        used_solvent.imass['l', 'Glucan'] = cellulose_mass*(1-solvolysis_parameters['Cellulose_retention']) # Dissolved cellulose assumed to be in liquid phase as solution with solvent
        used_biomass.imass['Glucan'] =  cellulose_mass*solvolysis_parameters['Cellulose_retention']
                                                               

        xylose_mass = used_biomass.imass['Xylan']
        used_solvent.imass['l', 'Xylan'] = xylose_mass * (1-solvolysis_parameters['Xylose_retention']) # Dissolved xylose assumed to be  liquid phase as solution with solvent
        used_biomass.imass['Xylan'] = xylose_mass * solvolysis_parameters['Xylose_retention']

        arabinan_mass = used_biomass.imass['Arabinan']
        used_solvent.imass['l', 'Arabinan'] = arabinan_mass * (1-solvolysis_parameters['Arabinan_retention'])

        used_biomass.imass['Arabinan'] = arabinan_mass * solvolysis_parameters['Arabinan_retention']
        
        mannan_mass = used_biomass.imass['Mannan']
        used_solvent.imass['l', 'Mannan'] = mannan_mass * (1-solvolysis_parameters['Mannan_retention']) 
        used_biomass.imass['Mannan'] = mannan_mass * solvolysis_parameters['Mannan_retention']

        galactan_mass = used_biomass.imass['Galactan']
        used_solvent.imass['l', 'Galactan'] = galactan_mass * (1-solvolysis_parameters['Galactan_retention']) 
        used_biomass.imass['Galactan'] = galactan_mass * solvolysis_parameters['Galactan_retention']
        

        # The temperature and pressure of the carbohydrate pulp is not changed here, I'm assuming I obtain the pulp at ambient conditons 
        # once RCF reaction is complete for downstream processing
        



    def _calculate_pressure_drop(self, bed_length):

        D = self.poplar_diameter                                # [m] poplar particle diameter
        rho = self.ins[1].rho                                   # [kg/m3] 
        mu = self.ins[1].get_property('mu', 'kg/m/s')           # [Pa s] methanol water viscosity 
        epsilon = self.void_frac                                # Void fraction 
        u = self.superficial_velocity                           # [m/s] superficial velocity  

        
        
        Re = (D*rho*u)/mu
        if Re/(1-epsilon) < 500: # Erun equation
            f = ((1-epsilon)/(epsilon**3))*(1.75+(150*(1-epsilon)/Re))
            dP = (f * ((rho*(u**2))/D)* bed_length)*1e-5                # [bar] 1e-5 converts Pa to bar
        elif 1000 < Re/(1-epsilon) < 5000: # Handley and Heggs
            f = ((1-epsilon)/(epsilon**3))*(1.24+(368*(1-epsilon)/Re))
            dP = (f * ((rho*(u**2))/D)* bed_length)*1e-5
        else: # Hicks equation which fits in Wentz and Thodos results for very high Re
            f = 6.8*(((1-epsilon)**1.2)/epsilon**3)*Re**-0.2
            dP = (f * ((rho*(u**2))/D) * bed_length)*1e-5
        return dP

        

    def _design(self):
        length, diameter, N_reactors, N_operating, total_volume = self._size_bed()   # Calling size bed function to determine diameter and length 
        


        
        
        
        cycle_time = self.tau + self.tau_0

        self.set_design_result('Diameter', 'ft', diameter)
        self.set_design_result('Length', 'ft', length)
        self.set_design_result('Reactor volume', 'm3', self.V_max)
        self.set_design_result('Total volume', 'm3', total_volume)
        self.set_design_result('Total beds', '', N_reactors)
        self.set_design_result('Beds in service', '', N_operating)
        self.set_design_result('Time on stream', 'hr', self.tau)
        self.set_design_result('Residence time', 'hr', self.tau_residence)
        self.set_design_result('Turnaround time', 'hr', self.tau_0)
        self.set_design_result('Batch time', 'hr', cycle_time)
        self.set_design_result('Biomass volume per bed', 'm3', self._V_biomass)
        self.set_design_result('Solvent volume per bed', 'm3', self._V_solvent)
        self.set_design_result('Instantaneous loading', 'L/kg', self._instantaneous_loading)
        self.set_design_result('Solvent loading', 'L/kg', self._loading)

        
        
        # Calculates weight based off pressure, diameter and length
        # Adds vcessel type wall thickness, vessel weight, diameter and length to dictionary
        # But diameter and length are already there because of set_design_result above
        
        self.design_results.update(
            self._vertical_vessel_design(    
                self.P*(1/6894.76),
                self.design_results['Diameter']*3.28084,
                self.design_results['Length']*3.28084
            )
        )
        
                            

        pressure_drop = self._calculate_pressure_drop(length)                  
        
        self.set_design_result('Pressure drop', 'bar', pressure_drop)
        self.pump_1.P = (self.P - self.ins[1].P) + (pressure_drop*1e5)
        self.pump_1.simulate()





    def _cost(self):
        design = self.design_results # Calling the dictionary used to store design results in design method above 

        baseline_purchase_costs = self.baseline_purchase_costs # Dictionary for storing baseline costs

        weight = design['Weight']  # weight parameter stores the value from the 'Weight' key in the design dictionnary
        
        N_reactors = design['Total beds']
        # Calculates the baseline purchase cost based off diameter length and weight
        baseline_purchase_costs.update( 
            self._vessel_purchase_cost(weight, design['Diameter'], design['Length'])
        )

        self.parallel['self'] = N_reactors # Used to create multiple of the same beds
        self.parallel['pump_1'] = 1 # Just one pump needed, valves will redirect to whichever bed is online


        
       
        """
        ---------
          
        Parameters that can be further fine-tuned based on industry/national lab data
        - Void fraction of poplar bed: Herein assumed 0.5, this is subject to change
        - Working volue fraction: Herein assumed 80%, but can change depending on how well mass transfer occurs in real reactors
        - V_max: Maximum volume of a single reactor, herein assumed as 600 m3 based on Bartling et al 2021 paper, but subject to change
        - residence time: Herein 2 hrs, but could change based on which regime is more limiting. 


        ----------

        """

    